###OBJETIVO

Utilizaremos SQL para transformar informações em indicadores capazes de responder perguntas relevantes de People Analytics. O raciocínio adotado será:

>**Dados disponíveis → Possibilidades de cruzamento → Pergunta de negócio → Indicador → Query SQL → Resultado → Análise**

###CARREGAMENTO DA BASE E PREPARAÇÃO PARA ANALISE SQL

- Importa o Pandas e o SQLite, ferramentas utilizadas para manipulação dos dados e execução das consultas SQL.
- Carrega a Base 3 diretamente do GitHub e armazena os dados em um DataFrame.
- Cria um banco SQLite em memória e transforma o DataFrame em uma `tabela chamada colaboradores`, que será utilizada para executar as queries SQL da análise.


>Dessa forma, a base consolidada já fica preparada para a exploração dos indicadores de People Analytics nas próximas etapas.

In [44]:
import pandas as pd
import sqlite3

# Carrega a Base 3 diretamente do GitHub
url_base3 = "https://raw.githubusercontent.com/isidoropinheiro/Projeto-People-Analytics/main/Documentos/Base%20de%20Dados/base3_dados_consolidados.xlsx"

df_base3 = pd.read_excel(url_base3)

# Cria o banco SQLite em memória
con = sqlite3.connect(":memory:")

# Cria a tabela para execução das queries SQL
df_base3.to_sql(
    "colaboradores",
    con,
    index=False,
    if_exists="replace"
)

print("Base carregada no SQLite com sucesso.")

Base carregada no SQLite com sucesso.


###DESCOBRINDO OS DADOS PARA IDENTIFICAR POSSIVEIS ANALISES

>Para estipular possiveis analises primeiro devemos descobrir o que temos a nosso favor. Nesse momento iremos analisar cada coluna da Base 3 para que possamos identificar possibilidades de cruzamento e perguntas de negócio.

In [45]:
#Lê todas as colunas
query = """
PRAGMA table_info(colaboradores);
"""

resultado = pd.read_sql_query(query, con)
resultado

,cid,name,type,notnull,dflt_value,pk
0,0,ID_Colaborador,TEXT,0,None,0
1,1,Area,TEXT,0,None,0
2,2,Cargo,TEXT,0,None,0
3,3,Senioridade,TEXT,0,None,0
4,4,Gestor,TEXT,0,None,0
5,5,Regiao,TEXT,0,None,0
6,6,Salario,REAL,0,None,0
7,7,Data_Admissao,TIMESTAMP,0,None,0
8,8,Data_Desligamento,TIMESTAMP,0,None,0
9,9,Motivo_Desligamento,TEXT,0,None,0


>Identificamos 13 colunas, com dados textuais, salariais e temporais.

- Com esses dados conseguimos ter a visão de:

| Grupo de análise | Indicadores | Outras análises possíveis |
|---|---|---|
| Movimentação de pessoas | Headcount, turnover, admissões e desligamentos | Evolução das entradas e saídas ao longo do tempo |
| Perfil organizacional | Quantidade e percentual de colaboradores por categoria | Distribuição por área, cargo, senioridade e região |
| Remuneração | Salário médio | Comparação salarial por área, cargo e senioridade |
| Análise de desligamentos | Taxa de desligamento, distribuição percentual dos motivos | Desligamentos por área, senioridade e gestor; tempo de casa na saída |
| Qualidade dos dados | Percentual de registros inconsistentes | Identificação e classificação de inconsistências pela `Flag_Qualidade` |
<br>

DIFERENÇA ENTRE CONCEITOS

- **Grupo de análise:** tema investigado.
- **Indicador:** medida calculada para quantificar um fenômeno.
- **Análise:** exploração ou comparação dos resultados para identificar padrões.

>Nem todas as análises da tabela são indicadores por si só. Por exemplo, comparar desligamentos por gestor é uma análise; a quantidade ou a taxa de desligamentos por gestor é um indicador.

- Com base nas possibilidades de análise identificadas na base de dados, foram definidas 15 análises exploratórias, cada uma acompanhada de sua respectiva query SQL, com o objetivo de investigar os dados, identificar padrões e construir uma visão mais ampla sobre o cenário organizacional sob a perspectiva de People Analytics.

###1 - ADMISSÕES

**Dados utilizados:** `Data_Admissao`

- Contabiliza as entradas de colaboradores em determinado período.

>**O que podemos descobrir:** quantas pessoas ingressaram na organização, como as admissões variam ao longo do tempo e quais períodos concentraram mais contratações.

In [46]:
# Admissões
query = """
SELECT
    strftime('%Y', Data_Admissao) AS Ano,
    COUNT(*) AS Total_Admissoes
FROM colaboradores
WHERE Data_Admissao IS NOT NULL
GROUP BY Ano
ORDER BY Ano;
"""

resultado = pd.read_sql_query(query, con)
resultado

,Ano,Total_Admissoes
0,2021,116
1,2022,87
2,2023,89
3,2024,86
4,2025,122


CONCLUSÃO QUERY ADMISSÕES

- 2025 teve o maior volume: 122 admissões.
- 2024 teve o menor volume: 86 admissões.
- Queda após 2021: as admissões passaram de 116 para 87 em 2022.
- Estabilidade entre 2022 e 2024: os números ficaram próximos, entre 86 e 89.
- Retomada em 2025: houve aumento de 36 admissões em relação a 2024, aproximadamente 41,9%.
>**Nova Hipótese para investigar:** o aumento de admissões em 2025 pode estar relacionado à expansão de determinadas áreas, ou à abertura de novas posições ou à reposição de profissionais desligados?

<br>


CÁLCULO DA VARIAÇÃO PERCENTUAL DAS ADMISSÕES



Para comparar 2024 e 2025, utilizamos os seguintes valores:

- 2024: 86 admissões (valor inicial).

- 2025: 122 admissões (valor final).

1. Calculamos a diferença entre os períodos: 122 - 86 = 36. Isso significa que houve 36 admissões a mais em 2025.

2. Dividimos essa diferença pelo valor inicial (36 / 86) para descobrir quanto o aumento representa em relação a 2024.

3. Multiplicamos por 100 para converter o resultado em porcentagem.

- Fórmula: `( (Valor final - Valor inicial) / Valor inicial ) * 100`

- Aplicação: `((122 - 86) / 86) * 100 ≈ 41,86%`

>**Interpretação:** o número de admissões em 2025 foi aproximadamente 41,86% maior que em 2024, passando de 86 para 122 registros.

<br>

---
**PENSANDO FORA DA CAIXA!**

---


>Se identificamos os períodos com mais admissões, podemos levantar estas hipóteses:
1. **Sazonalidade das contratações:** existem meses ou períodos do ano em que a organização contrata mais?
2. **Expansão organizacional:** os picos de admissão podem indicar crescimento da operação ou abertura de novas equipes?
3. **Concentração por área:** quais áreas concentram mais contratações em determinados períodos?
4. **Perfil das contratações:** quais cargos e níveis de senioridade são mais contratados em cada período?
5. **Reposição de profissionais:** as admissões acontecem próximas a períodos com mais desligamentos?
6. **Crescimento do Headcount:** as contratações superam os desligamentos, indicando aumento do quadro?
7. **Mudança na estrutura organizacional:** existem períodos em que determinadas áreas recebem mais profissionais do que outras?
8. **Remuneração das novas contratações:** qual é a média salarial dos profissionais admitidos em cada período?
9. **Concentração de admissões por gestor:** quais gestores recebem mais profissionais em determinados períodos?
>**Importante:** essas análises identificam padrões e associações. A base permite observar quando e onde as admissões acontecem, mas não comprova, sozinha, o motivo dos picos.

###2 - DESLIGAMENTOS

**Dados utilizados:** `Data_Desligamento`

- Contabiliza as saídas de colaboradores em determinado período.

>**O que podemos descobrir:** quantas pessoas saíram da organização e em quais períodos houve maior concentração de desligamentos.

In [47]:
# Desligamentos
query = """
SELECT
    strftime('%Y', Data_Desligamento) AS Ano,
    COUNT(*) AS Total_Desligamentos
FROM colaboradores
WHERE Data_Desligamento IS NOT NULL
GROUP BY Ano
ORDER BY Ano;
"""

resultado = pd.read_sql_query(query, con)
resultado

,Ano,Total_Desligamentos
0,2020,1
1,2021,4
2,2022,7
3,2023,19
4,2024,22
5,2025,32
6,2026,22
7,2027,31
8,2028,10
9,2029,5


CONCLUSÃO — QUERY DE DESLIGAMENTOS

- **2025 teve o maior volume de desligamentos entre 2020 e 2029:** 32 saídas em 2025 e 31 em 2027.
- **2020 teve o menor volume:** apenas 1 desligamento registrado.
- **Crescimento entre 2021 e 2025:** os desligamentos passaram de 4 para 32, um aumento de 700%.
- **Aumento progressivo entre 2022 e 2025:** os desligamentos passaram de 7 para 32.
- **Redução após 2027:** os registros caíram de 31 em 2027 para 10 em 2028 e 5 em 2029.

> **Nova hipótese para investigar:** o aumento de desligamentos entre 2022 e 2025 pode estar relacionado à expansão do quadro nos anos anteriores, à reposição de profissionais, a mudanças organizacionais ou à concentração de saídas em determinadas áreas?

<br>


CÁLCULO DA VARIAÇÃO PERCENTUAL DOS DESLIGAMENTOS


Para comparar 2021 e 2025, utilizamos os seguintes valores:

- **2021:** 4 desligamentos (valor inicial).
- **2025:** 32 desligamentos (valor final).

1. Calculamos a diferença entre os períodos: `32 - 4 = 28`. Isso significa que houve 28 desligamentos a mais em 2025.
2. Dividimos essa diferença pelo valor inicial (`28 / 4`) para descobrir quanto o aumento representa em relação a 2021.
3. Multiplicamos por 100 para converter o resultado em porcentagem.

- **Fórmula:** `( (Valor final - Valor inicial) / Valor inicial ) * 100`

- **Aplicação:** `((32 - 4) / 4) * 100 = 700%`

>**Interpretação:** o número de desligamentos em 2025 foi 700% maior que em 2021, passando de 4 para 32 registros.

<br>

---

PENSANDO FORA DA CAIXA!

---

>Se identificamos os períodos com mais desligamentos, podemos levantar as seguintes hipóteses:

1. **Sazonalidade dos desligamentos:** existem meses ou períodos do ano em que as saídas se concentram?

2. **Relação entre admissões e desligamentos:** os períodos com mais contratações também apresentam mais saídas?

3. **Reposição de profissionais:** as admissões realizadas em determinado período podem estar relacionadas à substituição de colaboradores desligados?

4. **Concentração por área:** quais áreas concentram mais desligamentos nos períodos de maior volume?

5. **Perfil dos desligamentos:** quais cargos e níveis de senioridade aparecem com maior frequência entre as saídas?

6. **Distribuição por gestor:** os desligamentos estão concentrados em determinadas estruturas de gestão?

7. **Motivos de desligamento:** quais motivos são mais frequentes nos anos com maior número de saídas?

8. **Tempo de casa:** os desligamentos ocorrem principalmente entre profissionais recém-admitidos ou entre colaboradores com mais tempo de organização?

9. **Impacto no Headcount:** as admissões foram suficientes para compensar os desligamentos ou houve redução do quadro?

10. **Remuneração dos desligados:** existe diferença entre a remuneração média dos colaboradores desligados e a dos colaboradores ativos?

> **Importante:** essas análises permitem identificar padrões e associações, mas não comprovam, isoladamente, as causas dos desligamentos.

###3 - DISTRIBUIÇÃO DE COLABORADORES POR AREA

**Dados utilizados:** `Area` e `Status`

- Identifica como os colaboradores ativos estão distribuídos entre as áreas da organização.

>**O que podemos descobrir:** quais áreas concentram mais colaboradores e como a força de trabalho está distribuída internamente.

In [48]:
# Distribuição Colaboradores por Area
query = """
SELECT
    Area,
    COUNT(*) AS Total_Colaboradores
FROM colaboradores
WHERE Status = 'Ativo'
GROUP BY Area
ORDER BY Total_Colaboradores DESC;
"""

resultado = pd.read_sql_query(query, con)
resultado

,Area,Total_Colaboradores
0,Operações,43
1,Marketing,43
2,Comercial,41
3,Produto,39
4,Dados,39
5,Jurídico,36
6,Recursos Humanos,35
7,Financeiro,31
8,Tecnologia,29


CONCLUSÃO — QUERY DE DISTRIBUIÇÃO DE COLABORADORES POR ÁREA

- **Operações e Marketing lideram:** ambas possuem 43 colaboradores ativos.
- **Comercial aparece em seguida:** com 41 colaboradores ativos.
- **Produto e Dados:** possuem 39 colaboradores ativos cada.
- **Jurídico:** conta com 36 colaboradores ativos.
- **Recursos Humanos:** possui 35 colaboradores ativos.
- **Financeiro:** conta com 31 colaboradores ativos.
- **Tecnologia apresenta o menor quadro:** com 29 colaboradores ativos.

>**Interpretação:** A distribuição mostra que Operações e Marketing possuem os maiores quadros ativos, enquanto Tecnologia possui o menor. Entretanto, a diferença entre a maior e a menor área é de apenas 14 colaboradores, indicando uma distribuição relativamente equilibrada em termos absolutos.

<br>

---

PENSANDO FORA DA CAIXA!

---


1. **Dimensionamento das equipes:** o número de colaboradores é compatível com a demanda e as responsabilidades de cada área?
2. **Relação entre admissões e desligamentos:** quais áreas cresceram ou reduziram seu quadro ao longo do tempo?
3. **Distribuição de senioridade:** as áreas com mais colaboradores também concentram profissionais mais experientes?
4. **Remuneração por área:** áreas com quadros menores apresentam salários médios diferentes?
5. **Desligamentos por área:** as áreas com mais colaboradores também registram mais desligamentos?
6. **Concentração por gestor:** como os colaboradores estão distribuídos entre os gestores de cada área?

> **Importante:** a quantidade de colaboradores, isoladamente, não permite concluir se uma equipe está superdimensionada ou subdimensionada. Para isso, precisaríamos relacionar o quadro com demanda, produtividade, responsabilidades e estrutura organizacional.

###4 - DISTRIBUIÇÃO POR SENIORIDADE

**Dados utilizados:** `Senioridade` e `Status`

- Analisa a composição do quadro ativo por nível de senioridade.

>**O que podemos descobrir:** quais níveis de senioridade predominam na organização e como os profissionais estão distribuídos entre os diferentes níveis.

In [49]:
# Distribuição por Senioridade
query = """
SELECT
    Senioridade,
    COUNT(*) AS Total_Colaboradores
FROM colaboradores
WHERE Status = 'Ativo'
GROUP BY Senioridade
ORDER BY Total_Colaboradores DESC;
"""

resultado = pd.read_sql_query(query, con)
resultado

,Senioridade,Total_Colaboradores
0,Pleno,126
1,Júnior,90
2,Sênior,72
3,Estágio,28
4,Especialista,20


CONCLUSÃO — QUERY DE DISTRIBUIÇÃO POR SENIORIDADE

- **Pleno lidera:** com 126 colaboradores ativos, é o nível de senioridade mais representativo da organização.
- **Júnior aparece em segundo lugar:** com 90 colaboradores ativos.
- **Sênior:** conta com 72 colaboradores ativos.
- **Estágio:** possui 28 colaboradores ativos.
- **Especialista apresenta o menor volume:** com 20 colaboradores ativos.

>**Interpretação:** A maior concentração está nos níveis Pleno e Júnior. Juntos, representam 216 dos 336 colaboradores ativos, aproximadamente 64,29% do quadro.

<br>

CÁLCULO DO PERCENTUAL DOS COLABORADORES PLENO E JÚNIOR


Para identificar a participação desses dois níveis no quadro ativo, utilizamos os seguintes valores:

- **Pleno:** 126 colaboradores.
- **Júnior:** 90 colaboradores.
- **Total de colaboradores ativos:** 336.

1. Somamos os colaboradores Pleno e Júnior: `126 + 90 = 216`.
2. Dividimos esse total pelo número de colaboradores ativos: `216 / 336 ≈ 0,6429`.
3. Multiplicamos por 100 para converter o resultado em porcentagem: `0,6429 * 100 ≈ 64,29%`.

- **Fórmula:** `( (Pleno + Júnior) / Total de colaboradores ativos ) * 100`
- **Aplicação:** `((126 + 90) / 336) * 100 ≈ 64,29%`


<br>

---

PENSANDO FORA DA CAIXA!

---

1. **Concentração de experiência:** a predominância de profissionais Pleno e Júnior pode indicar uma estrutura com forte presença de profissionais em desenvolvimento e consolidação de carreira?
2. **Distribuição por área:** quais áreas concentram mais profissionais Júnior, Pleno ou Sênior?
3. **Remuneração por senioridade:** como o salário médio varia entre os níveis?
4. **Desligamentos por senioridade:** quais níveis concentram mais saídas e como isso se compara à participação de cada nível no quadro?
5. **Desenvolvimento interno:** os dados de admissão e desligamento permitem investigar a permanência dos profissionais em cada nível?
6. **Estrutura de especialistas:** em quais áreas estão os profissionais Especialistas e qual é sua participação no quadro?

> **Importante:** a distribuição mostra a composição atual da base, mas não permite concluir, isoladamente, se existe falta de profissionais Sênior ou Especialistas. Para isso, precisaríamos conhecer as necessidades e a estrutura esperada de cada área.

###5 - SALÁRIO MÉDIO

**Dados utilizados:** `Salario`

- Calcula a média salarial geral ou de grupos específicos de colaboradores.

>**O que podemos descobrir:** qual é a remuneração média da base e como ela varia quando segmentamos os colaboradores por características organizacionais.

In [50]:
# Salario Medio
query = """
SELECT
    ROUND(AVG(Salario), 2) AS Salario_Medio
FROM colaboradores
WHERE Status = 'Ativo';
"""

resultado = pd.read_sql_query(query, con)
resultado

,Salario_Medio
0,6255.64


CONCLUSÃO — QUERY DE SALÁRIO MÉDIO

- **Salário médio dos colaboradores ativos:** R$ 6.255,64.

>**Interpretação:** A média salarial dos 336 colaboradores ativos é de **R$ 6.255,64**. Esse valor representa a média aritmética dos salários registrados na base para esse grupo.

A média serve como referência inicial para comparar a remuneração entre áreas, cargos e níveis de senioridade. Entretanto, não significa que a maioria dos colaboradores receba exatamente esse valor.

<br>

---

PENSANDO FORA DA CAIXA!

---

1. **Diferenças entre áreas:** quais áreas apresentam salários médios acima ou abaixo da média geral?
2. **Remuneração por senioridade:** como o salário médio varia entre Júnior, Pleno, Sênior e Especialista?
3. **Distribuição salarial:** os salários estão concentrados próximos da média ou existem valores muito altos ou baixos que influenciam o resultado?
4. **Remuneração por cargo:** quais cargos apresentam as maiores e menores médias salariais?
5. **Composição do quadro:** a média geral pode ser influenciada pela concentração de colaboradores em determinados cargos ou níveis de senioridade?

> **Importante:** a média salarial, isoladamente, não permite concluir se a remuneração é competitiva ou justa. Para isso, seria necessário comparar os valores com critérios internos, responsabilidades dos cargos ou referências de mercado.

###6 - DISTRIBUIÇÃO SALARIAL MÉDIA POR AREA

**Dados utilizados:** `Area` e `Salario`

- Compara a remuneração média entre as áreas.

>**O que podemos descobrir:** quais áreas apresentam maiores ou menores médias salariais e se existem diferenças de remuneração entre elas. A comparação, isoladamente, não explica as causas dessas diferenças.

In [51]:
# Distribuição Salarial Média por Area
query = """
SELECT
    Area,
    ROUND(AVG(Salario), 2) AS Salario_Medio
FROM colaboradores
WHERE Status = 'Ativo'
GROUP BY Area
ORDER BY Salario_Medio DESC;
"""

resultado = pd.read_sql_query(query, con)
resultado

,Area,Salario_Medio
0,Operações,7163.91
1,Dados,6832.80
2,Comercial,6641.21
3,Financeiro,6411.95
4,Jurídico,6330.68
5,Recursos Humanos,6273.54
6,Produto,5685.21
7,Tecnologia,5485.70
8,Marketing,5302.85


CONCLUSÃO — QUERY DE DISTRIBUIÇÃO SALARIAL POR ÁREA

- **Operações apresenta a maior média salarial:** R$ 7.163,91.

- **Dados ocupa a segunda posição:** R$ 6.832,80.

- **Comercial aparece em terceiro:** R$ 6.641,21.

- **Marketing apresenta a menor média salarial:** R$ 5.302,85.

- **Tecnologia também apresenta uma média relativamente baixa:** R$ 5.485,70.

>**Interpretação:** Existe uma diferença de R$ 1.861,06 entre a maior média salarial (Operações) e a menor (Marketing).

<br>


CÁLCULO DA DIFERENÇA PERCENTUAL ENTRE AS MÉDIAS SALARIAIS


Para comparar Operações e Marketing, utilizamos:

- **Operações:** R$ 7.163,91 (valor de referência superior).

- **Marketing:** R$ 5.302,85 (valor de referência inferior).
<br>

1. Calculamos a diferença: `7.163,91 - 5.302,85 = 1.861,06`.
2. Dividimos a diferença pela média de Marketing: `1.861,06 / 5.302,85 ≈ 0,3509`.
3. Multiplicamos por 100 para converter em porcentagem.

- **Fórmula:** `((Maior valor - Menor valor) / Menor valor) * 100`
- **Aplicação:** `((7.163,91 - 5.302,85) / 5.302,85) * 100 ≈ 35,09%`

> **Interpretação:** a média salarial de Operações é aproximadamente 35,09% maior que a de Marketing.

<br>

---

PENSANDO FORA DA CAIXA!

---

1. **Influência da senioridade:** Operações possui salários médios maiores porque concentra mais profissionais Sênior ou Especialistas?
2. **Composição dos cargos:** as diferenças salariais estão relacionadas aos cargos existentes em cada área?
3. **Distribuição salarial interna:** as médias refletem o padrão salarial da maioria dos colaboradores ou são influenciadas por alguns salários elevados?
4. **Comparação com a média geral:** quais áreas estão acima ou abaixo do salário médio geral de R$ 6.255,64?
5. **Desligamentos e remuneração:** áreas com salários médios menores também concentram mais desligamentos?
6. **Equidade salarial:** existem diferenças entre áreas que merecem investigação após considerar cargo e senioridade?

> **Importante:** a diferença entre as médias não comprova desigualdade salarial ou problemas de remuneração. Para interpretar os resultados, precisamos considerar as funções, a senioridade e a composição de cada área.

###7 - DISTRIBUIÇÃO SALARIAL POR SENIORIDADE

**Dados utilizados:** `Senioridade` e `Salario`

- Compara a remuneração média entre os níveis de senioridade.

>**O que podemos descobrir:** como a remuneração varia entre os níveis e se profissionais de maior senioridade apresentam, em média, salários mais elevados.

In [52]:
# Distribuição Salarial por senioridade
query = """
SELECT
    Senioridade,
    ROUND(AVG(Salario), 2) AS Salario_Medio
FROM colaboradores
WHERE Status = 'Ativo'
GROUP BY Senioridade
ORDER BY Salario_Medio DESC;
"""
resultado = pd.read_sql_query(query, con)
resultado

,Senioridade,Salario_Medio
0,Especialista,14460.32
1,Sênior,9808.68
2,Pleno,5902.11
3,Júnior,3557.56
4,Estágio,1522.10


CONCLUSÃO — QUERY DE SALÁRIO MÉDIO POR SENIORIDADE

- **Especialista:** apresenta a maior média salarial, de R$ 14.460,32.

- **Sênior:** segunda maior média, com R$ 9.808,68.

- **Pleno:** média de R$ 5.902,11.

- **Júnior:** média de R$ 3.557,56.

- **Estágio:** apresenta a menor média, de R$ 1.522,10.


>**Interpretação:** Os resultados mostram uma progressão salarial média conforme aumenta o nível de senioridade. A diferença entre Especialista e Estágio é de R$ 12.938,22, indicando uma amplitude significativa entre os níveis extremos.

<br>

---

PENSANDO FORA DA CAIXA!

---

- **Progressão de carreira:** investigar como a remuneração evolui entre Júnior, Pleno, Sênior e Especialista.
- **Composição das áreas:** verificar se a média de Especialistas é influenciada pela concentração desses profissionais em áreas com salários mais elevados.
- **Equidade salarial:** comparar salários de profissionais da mesma senioridade e área para identificar diferenças relevantes.
- **Retenção:** cruzar senioridade e desligamentos para investigar quais níveis apresentam maior saída proporcional ao tamanho de cada grupo.
- **Desenvolvimento interno:** avaliar se a progressão de senioridade acompanha o desenvolvimento profissional e a evolução salarial.

>**Atenção:** a média não representa o salário de todos os profissionais daquele nível. Para avaliar equidade ou competitividade, também seria necessário analisar a distribuição dos salários e considerar os cargos e as áreas.

###8 - MOTIVOS DE DESLIGAMENTO

**Dados utilizados:** `Motivo_Desligamento`

- Identifica a frequência dos diferentes motivos registrados para as saídas.

>**O que podemos descobrir:** quais motivos aparecem com maior frequência entre os desligamentos registrados. Isso ajuda a identificar padrões que podem ser investigados posteriormente.

In [53]:
# Motivo de Desligamento
query = """
SELECT
    Motivo_Desligamento,
    COUNT(*) AS Total_Desligamentos
FROM colaboradores
WHERE Status = 'Desligado'
  AND Motivo_Desligamento IS NOT NULL
GROUP BY Motivo_Desligamento
ORDER BY Total_Desligamentos DESC;
"""
resultado = pd.read_sql_query(query, con)
resultado

,Motivo_Desligamento,Total_Desligamentos
0,Pedido de demissão,110
1,Desempenho,10
2,Fim de contrato,9
3,Reestruturação,7
4,Mudança de carreira,6
5,Outro,3
6,Salário,2


CONCLUSÃO — QUERY DE MOTIVOS DE DESLIGAMENTO

- **Pedido de demissão:** principal motivo registrado, com 110 desligamentos.
- **Desempenho:** segundo motivo mais frequente, com 10 registros.
- **Fim de contrato:** 9 desligamentos.
- **Reestruturação:** 7 desligamentos.
- **Mudança de carreira:** 6 desligamentos.
- **Outro:** 3 desligamentos.
- **Salário:** 2 desligamentos.

> **Interpretaçã:** O pedido de demissão representa a grande maioria dos motivos registrados. Considerando os 147 desligamentos identificados na consulta, foram 110 pedidos de demissão, aproximadamente **74,83% do total**.

<br>

CÁLCULO DO PERCENTUAL


1. **Total de pedidos de demissão:** 110.
2. **Total de desligamentos considerados na consulta:** 147, resultado da soma de todos os motivos registrados.
3. Dividimos os 110 pedidos de demissão pelos 147 desligamentos para descobrir qual proporção eles representam: `110 ÷ 147 = 0,7483`.
4. Multiplicamos o resultado por 100 para fazer a converção em percentual: `0,7483 × 100 = 74,83%`.

- **Fórmula:** `(Pedidos de demissão ÷ Total de desligamentos) × 100`

>**Interpretação:** os pedidos de demissão representam aproximadamente **74,83% dos desligamentos considerados na consulta**.

<br>

---

PENSANDO FORA DA CAIXA!

---


- **Retenção:** investigar por que tantos colaboradores pedem demissão, cruzando os dados com área, senioridade, salário e tempo de casa.
- **Liderança:** verificar se os pedidos de demissão se concentram em determinadas áreas ou sob determinados gestores.
- **Remuneração:** embora apenas dois registros indiquem salário como motivo, investigar se a remuneração aparece indiretamente em outras razões de saída.
- **Desenvolvimento profissional:** cruzar pedidos de demissão com senioridade para identificar possíveis padrões entre níveis de carreira.
- **Tempo de casa:** analisar se as saídas ocorrem principalmente nos primeiros meses ou após períodos mais longos na empresa.

>**Atenção:** o motivo registrado não explica necessariamente toda a decisão de saída. Além disso, os resultados mostram quantidades absolutas; para comparar áreas ou níveis, será necessário considerar também o tamanho de cada grupo.

###9 - DESLIGAMENTOS POR AREA

**Dados utilizados:** `Area` e `Data_Desligamento`

- Compara a quantidade de desligamentos entre as áreas em determinado período.

>**O que podemos descobrir:** quais áreas concentram mais saídas. Para comparar áreas de tamanhos diferentes, também pode ser útil calcular taxas de desligamento, evitando conclusões baseadas apenas em números absolutos.

In [54]:
# Desligamentos por Area
query = """
SELECT
    Area,
    COUNT(*) AS Total_Desligamentos
FROM colaboradores
WHERE Data_Desligamento IS NOT NULL
GROUP BY Area
ORDER BY Total_Desligamentos DESC;
"""
resultado = pd.read_sql_query(query, con)
resultado

,Area,Total_Desligamentos
0,Recursos Humanos,23
1,Comercial,19
2,Financeiro,18
3,Dados,18
4,Tecnologia,17
5,Operações,16
6,Marketing,16
7,Produto,13
8,Jurídico,13


CONCLUSÃO — QUERY DE DESLIGAMENTOS POR ÁREA

- **Recursos Humanos:** lidera em quantidade de desligamentos, com 23 registros.
- **Comercial:** apresenta 19 desligamentos.
- **Financeiro e Dados:** registram 18 desligamentos cada.
- **Tecnologia:** apresenta 17 desligamentos.
- **Operações e Marketing:** registram 16 desligamentos cada.
- **Produto e Jurídico:** apresentam os menores números, com 13 desligamentos cada.

> **Interpretação:** Os resultados mostram que Recursos Humanos concentra a maior quantidade absoluta de desligamentos, enquanto Produto e Jurídico apresentam os menores volumes. A diferença entre essas áreas é de 10 desligamentos.

<br>

---

PENSANDO FORA DA CAIXA!

---

- **Taxa de desligamento por área:** comparar os desligamentos com o total de colaboradores de cada área para avaliar o impacto proporcional.
- **Retenção em Recursos Humanos:** investigar se as saídas estão relacionadas a senioridade, remuneração, tempo de casa ou liderança.
- **Padrões de saída:** cruzar as áreas com os motivos de desligamento para identificar possíveis concentrações.
- **Perfil dos desligados:** avaliar quais níveis de senioridade e cargos apresentam mais saídas em cada área.
- **Admissões versus desligamentos:** verificar se as contratações compensaram as saídas ou se houve redução do quadro.

>**Atenção:** a consulta considera todos os registros com `Data_Desligamento` preenchida, sem limitar um período específico. Os números são absolutos e, isoladamente, não permitem concluir qual área apresenta maior rotatividade.

###10 - DESLIGAMENTOS POR SENIORIDADE

**Dados utilizados:** `Senioridade` e `Data_Desligamento`

- Analisa a quantidade de desligamentos em cada nível de senioridade.

> **O que podemos descobrir:** quais níveis concentram mais saídas e se determinados grupos merecem uma investigação mais aprofundada. A quantidade de desligamentos, sozinha, não demonstra maior propensão a sair.

In [55]:
#Desligamentos por Senioridade
query = """
SELECT
    Senioridade,
    COUNT(*) AS Total_Desligamentos
FROM colaboradores
WHERE Data_Desligamento IS NOT NULL
GROUP BY Senioridade
ORDER BY Total_Desligamentos DESC;
"""
resultado = pd.read_sql_query(query, con)
resultado

,Senioridade,Total_Desligamentos
0,Pleno,47
1,Sênior,46
2,Júnior,39
3,Estágio,16
4,Especialista,5


CONCLUSÃO — QUERY DE DESLIGAMENTOS POR SENIORIDADE

- **Pleno:** lidera em quantidade de desligamentos, com 47 registros.
- **Sênior:** apresenta 46 desligamentos, praticamente o mesmo volume de Pleno.
- **Júnior:** registra 39 desligamentos.
- **Estágio:** apresenta 16 desligamentos.
- **Especialista:** registra o menor volume, com 5 desligamentos.

>**Interpretação:** Os níveis Pleno e Sênior concentram os maiores números absolutos de desligamentos, somando 93 registros. Esse resultado indica um ponto de investigação, mas não significa, por si só, que esses profissionais tenham maior propensão a sair.

<br>

---

PENSANDO FORA DA CAIXA!

---

- **Taxa de desligamento por senioridade:** comparar os desligamentos com o total de profissionais de cada nível, considerando o tamanho de cada grupo.
- **Retenção de profissionais experientes:** investigar fatores relacionados à remuneração, oportunidades de crescimento e tempo de casa nos níveis Pleno e Sênior.
- **Motivos de saída:** cruzar senioridade e motivo de desligamento para identificar quais razões aparecem com maior frequência em cada nível.
- **Distribuição por área:** verificar se os desligamentos de Plenos e Seniores estão concentrados em áreas específicas.
- **Desenvolvimento de carreira:** avaliar se existem oportunidades de progressão suficientes para profissionais que buscam avançar na carreira.

>**Atenção:** a consulta apresenta quantidades absolutas, sem considerar o tamanho de cada nível de senioridade. Para comparar a propensão relativa a sair, será necessário calcular taxas de desligamento por grupo.

###11 - DESLIGAMENTOS POR GESTOR

**Dados utilizados:** `Gestor` e `Data_Desligamento`

- Compara a distribuição de desligamentos entre os gestores.

>**O que podemos descobrir:** quais estruturas de gestão concentram mais saídas e onde pode ser pertinente aprofundar a investigação. Os resultados precisam considerar o tamanho das equipes e não comprovam, isoladamente, que um gestor seja a causa dos desligamentos.

In [56]:
# Desligamentos por Gestor
query = """
SELECT
    Gestor,
    COUNT(*) AS Total_Desligamentos
FROM colaboradores
WHERE Data_Desligamento IS NOT NULL
GROUP BY Gestor
ORDER BY Total_Desligamentos DESC;
"""
resultado = pd.read_sql_query(query, con)
resultado

,Gestor,Total_Desligamentos
0,Gestor_017,10
1,Gestor_030,9
2,Gestor_023,9
3,Gestor_013,9
4,Gestor_027,8
5,Gestor_008,8
6,Gestor_019,7
7,Gestor_018,6
8,Gestor_003,6
9,Gestor_025,5


CONCLUSÃO — QUERY DE DESLIGAMENTOS POR GESTOR

- **Gestor_017:** apresenta a maior quantidade de desligamentos, com 10 registros.
- **Gestor_030, Gestor_023 e Gestor_013:** registram 9 desligamentos cada.
- **Gestor_027 e Gestor_008:** apresentam 8 desligamentos cada.
- Os demais gestores listados apresentam entre 3 e 7 desligamentos.

>**Interpretação:** Os resultados mostram que o Gestor_017 possui o maior número absoluto de desligamentos. Entretanto, a diferença para os gestores seguintes é pequena: apenas um desligamento a mais. Portanto, não é possível concluir, com esses dados isolados, que exista um problema específico de liderança.

<br>

---

PENSANDO FORA DA CAIXA!

---

- **Tamanho das equipes:** comparar os desligamentos com o total de colaboradores de cada gestor para calcular taxas proporcionais.
- **Motivos de desligamento:** investigar se determinados motivos se concentram em equipes específicas.
- **Senioridade:** verificar quais níveis profissionais apresentam mais saídas sob cada gestor.
- **Tempo de casa:** avaliar se os desligamentos ocorrem principalmente entre profissionais recém-contratados ou mais experientes.
- **Histórico de movimentação:** comparar admissões e desligamentos por equipe para entender se as saídas estão sendo compensadas por novas contratações.

>**Atenção:** a consulta considera todos os registros com `Data_Desligamento` preenchida. Como não há um período delimitado nem o tamanho das equipes nessa análise, os resultados representam volumes absolutos, e não taxas de rotatividade ou avaliações de desempenho dos gestores.

###12 - TEMPO DE CASA NOS DESLIGAMENTOS

**Dados utilizados:** `Tempo_de_Casa` e `Data_Desligamento`

- Analisa quanto tempo os colaboradores permaneceram na organização antes de sair.

> **O que podemos descobrir:** em quais momentos da trajetória profissional os desligamentos se concentram, como nos primeiros meses ou após vários anos de vínculo.

In [57]:
#Tempo de Casa nos Desligamentos
query = """
SELECT
    Tempo_de_Casa,
    COUNT(*) AS Total_Desligamentos
FROM colaboradores
WHERE Data_Desligamento IS NOT NULL
  AND Tempo_de_Casa IS NOT NULL
GROUP BY Tempo_de_Casa
ORDER BY Tempo_de_Casa;
"""
resultado = pd.read_sql_query(query, con)
resultado

,Tempo_de_Casa,Total_Desligamentos
0,1 ano,2
1,"1 ano, 1 mês",2
2,"1 ano, 10 meses",4
3,"1 ano, 11 meses",3
4,"1 ano, 2 meses",2
5,"1 ano, 4 meses",2
6,"1 ano, 5 meses",4
7,"1 ano, 6 meses",3
8,"1 ano, 7 meses",6
9,"1 ano, 8 meses",4


CONCLUSÃO — QUERY DE TEMPO DE CASA NOS DESLIGAMENTOS

- **2 anos e 1 mês:** apresenta a maior frequência, com 9 desligamentos.
- **1 ano e 7 meses, 2 anos, 2 anos e 4 meses e 6 meses:** registram 6 desligamentos cada.
- **2 anos e 8 meses, 2 anos e 9 meses e 3 anos e 1 mês:** apresentam 5 desligamentos cada.
- Os demais períodos apresentam entre 1 e 4 desligamentos.

>**Interpretação:** Os resultados mostram uma concentração de saídas entre aproximadamente 1 e 3 anos de vínculo, com destaque para o período de 2 anos e 1 mês. Também há desligamentos nos primeiros meses, incluindo 6 registros aos 6 meses de casa.

<br>

---

PENSANDO FORA DA CAIXA

---

- **Retenção inicial:** investigar os desligamentos ocorridos nos primeiros 6 meses, cruzando-os com área, cargo, senioridade e motivo de saída.
- **Marcos de carreira:** verificar se as saídas entre 1 e 3 anos estão relacionadas à falta de oportunidades de crescimento ou progressão salarial.
- **Liderança:** cruzar tempo de casa e gestor para verificar se existem concentrações relevantes em determinadas equipes.
- **Motivos de desligamento:** analisar se os pedidos de demissão predominam em períodos específicos do vínculo.
- **Experiência do colaborador:** investigar se integração, acompanhamento inicial, feedbacks e desenvolvimento podem ajudar a explicar os padrões observados.

>**Atenção:** essa consulta mostra a quantidade de desligamentos em cada tempo de casa, mas não calcula a probabilidade de saída em cada período. Para isso, seria necessário considerar também quantos colaboradores permanecem na empresa em cada faixa de tempo de casa.

<br>
ATENÇÃO EM RELAÇÃO A QUESTÃO DE DADOS!

- **Observação sobre os dados:** a tabela apresenta o valor `3 anos, 12 meses`. Esse registro merece validação, pois 12 meses equivalem a mais um ano.
> Também vale confirmar se `Tempo_de_Casa` foi calculado de forma consistente para todos os desligados.

###13 - DISTRIBUIÇÃO GEOGRÁFICA

**Dados utilizados:** `Regiao` e `Status`

- Analisa como os colaboradores ativos estão distribuídos entre as regiões registradas.

>**O que podemos descobrir:** quais regiões concentram mais colaboradores e como a força de trabalho está distribuída geograficamente.

In [58]:
# Distribuição Geografica
query = """
SELECT
    Regiao,
    COUNT(*) AS Total_Colaboradores
FROM colaboradores
WHERE Status = 'Ativo'
GROUP BY Regiao
ORDER BY Total_Colaboradores DESC;
"""
resultado = pd.read_sql_query(query, con)
resultado

,Regiao,Total_Colaboradores
0,PR,55
1,SP,50
2,SC,41
3,RS,40
4,RJ,40
5,BA,40
6,DF,36
7,MG,34


CONCLUSÃO — QUERY DE DISTRIBUIÇÃO GEOGRÁFICA

- **Paraná (PR):** concentra o maior número de colaboradores ativos, com 55.
- **São Paulo (SP):** aparece em segundo lugar, com 50 colaboradores.
- **Santa Catarina (SC):** registra 41 colaboradores.
- **Rio Grande do Sul (RS), Rio de Janeiro (RJ) e Bahia (BA):** apresentam 40 colaboradores cada.
- **Distrito Federal (DF):** registra 36 colaboradores.
- **Minas Gerais (MG):** apresenta o menor número, com 34 colaboradores.

>**Interpretação:** Os resultados mostram que o Paraná concentra a maior quantidade de colaboradores ativos, enquanto Minas Gerais apresenta a menor. A diferença entre as duas regiões é de 21 colaboradores.

<br>

---

PENSANDO FORA DA CAIXA!

---

- **Concentração geográfica:** investigar se a distribuição acompanha a localização de escritórios, clientes ou operações da empresa.
- **Recrutamento:** avaliar quais regiões oferecem maior disponibilidade de profissionais para futuras contratações.
- **Rotatividade regional:** cruzar região e desligamentos para identificar onde ocorrem mais saídas.
- **Remuneração:** comparar salários médios por região para investigar diferenças remuneratórias.
- **Planejamento de pessoas:** utilizar a distribuição geográfica para apoiar decisões sobre contratação, estrutura e alocação de equipes.

>**Atenção:** a consulta mostra quantidades absolutas de colaboradores ativos por região. Ela não permite concluir, isoladamente, se há concentração excessiva ou insuficiência de profissionais em alguma localidade.

###14 - COMPOSIÇÃO DO QUADRO POR CARGO

**Dados utilizados:** `Cargo` e `Status`

- Identifica a quantidade de colaboradores ativos em cada cargo.

>**O que podemos descobrir:** quais cargos são mais frequentes na organização e como as funções estão distribuídas no quadro de pessoal.

In [59]:
# Composição do Quadro Por Cargo
query = """
SELECT
    Cargo,
    COUNT(*) AS Total_Colaboradores
FROM colaboradores
WHERE Status = 'Ativo'
GROUP BY Cargo
ORDER BY Total_Colaboradores DESC;
"""
resultado = pd.read_sql_query(query, con)
resultado

,Cargo,Total_Colaboradores
0,Engenheiro de Dados,48
1,Desenvolvedor,48
2,Coordenador,42
3,Assistente,42
4,Especialista,40
5,Consultor,40
6,Gerente,38
7,Analista,38


CONCLUSÃO — QUERY DE COMPOSIÇÃO DO QUADRO POR CARGO

- **Engenheiro de Dados e Desenvolvedor:** lideram a distribuição, com 48 colaboradores ativos cada.
- **Coordenador e Assistente:** apresentam 42 colaboradores cada.
- **Especialista e Consultor:** registram 40 colaboradores cada.
- **Gerente e Analista:** apresentam 38 colaboradores cada.

>**Interpretação:** Os resultados mostram uma distribuição relativamente equilibrada entre os oito cargos listados. A diferença entre os cargos com maior e menor quantidade é de apenas 10 colaboradores.

<br>

---

PENSANDO FORA DA CAIXA!

---

- **Estrutura organizacional:** avaliar se a quantidade de coordenadores e gerentes é compatível com o tamanho das equipes e as necessidades de gestão.
- **Capacidade técnica:** investigar se a quantidade de Engenheiros de Dados e Desenvolvedores atende à demanda de projetos e produtos.
- **Distribuição por área:** cruzar cargo e área para entender onde cada função está concentrada.
- **Remuneração:** comparar salários médios por cargo para investigar diferenças e possíveis distorções.
- **Desenvolvimento de carreira:** analisar a relação entre cargo e senioridade para compreender a distribuição dos níveis profissionais em cada função.

>**Atenção:** a quantidade de pessoas por cargo não demonstra, isoladamente, se o quadro está dimensionado corretamente. Para avaliar isso, seria necessário considerar demanda de trabalho, estrutura das equipes e objetivos do negócio.

###15 - QUALIDADE DOS DADOS

**Dados utilizados:** `Flag_Qualidade`

- Identifica e classifica registros sinalizados por possíveis inconsistências.

>**O que podemos descobrir:** quantos registros precisam de revisão, quais tipos de inconsistência foram identificados e quais indicadores podem ser afetados por problemas na base.

In [60]:
# Qualidade dos Dados
query = """
SELECT
    Flag_Qualidade,
    COUNT(*) AS Total_Registros
FROM colaboradores
GROUP BY Flag_Qualidade
ORDER BY Total_Registros DESC;
"""
resultado = pd.read_sql_query(query, con)
resultado

,Flag_Qualidade,Total_Registros
0,OK,474
1,Motivo sem data,11
2,Desligamento anterior à admissão,5
3,Data sem motivo,5
4,Admissão e desligamento na mesma data,4
5,Data sem motivo; Admissão e desligamento na me...,1


CONCLUSÃO — QUERY DE QUALIDADE DOS DADOS

- **OK:** 474 registros não apresentam inconsistências sinalizadas.
- **Motivo sem data:** 11 registros possuem motivo de desligamento sem a data correspondente.
- **Desligamento anterior à admissão:** 5 registros apresentam uma inconsistência cronológica.
- **Data sem motivo:** 5 registros possuem data de desligamento sem motivo informado.
- **Admissão e desligamento na mesma data:** 4 registros apresentam datas coincidentes.
- **Múltiplas inconsistências:** 1 registro possui mais de uma inconsistência sinalizada.

>**Interpretação:** No total, **474 dos 500 registros estão classificados como OK**, enquanto 26 apresentam algum tipo de sinalização de qualidade.

<br>


CÁLCULO DO PERCENTUAL DE REGISTROS SINALIZADOS


1. **Total de registros:** 500.
2. **Registros sinalizados:** 26.
3. **Divisão:** `26 ÷ 500 = 0,052`.
4. **Conversão para percentual:** `0,052 × 100 = 5,2%`.

- **Fórmula:** `(Registros sinalizados ÷ Total de registros) × 100`

> **Resultado:** 5,2% dos registros apresentam possíveis inconsistências.

<br>

---

PENSANDO FORA DA CAIXA!

---

- **Confiabilidade dos indicadores:** inconsistências nas datas podem afetar cálculos de tempo de casa e análises de desligamentos.
- **Integridade do histórico:** verificar registros com desligamento anterior à admissão antes de utilizá-los em análises temporais.
- **Padronização dos processos:** estabelecer regras para evitar desligamentos sem data ou sem motivo registrado.
- **Priorização das correções:** revisar primeiro os registros com múltiplas inconsistências e os erros cronológicos.
- **Prevenção:** criar validações para identificar inconsistências durante o preenchimento ou a atualização da base.

>**Atenção:** as flags identificam registros que precisam de revisão, mas não confirmam necessariamente que todos estejam incorretos. O registro sinalizado deve ser validado antes de qualquer correção.

###FINALIZAÇÃO DA EXPLORAÇÃO DE BASE DE DADOS

>**Indicadores Analisados:** Com base nos resultados das 15 queries exploratórias, conseguimos responder às seguintes perguntas de negócio:

<br>

1. Movimentação de pessoas
- Quantas admissões ocorreram em cada ano?
- Quantos desligamentos ocorreram em cada ano?
<br>

2. Composição organizacional
- Como os colaboradores ativos estão distribuídos entre as áreas?
- Como os colaboradores ativos estão distribuídos por cargo e senioridade?
- Como os colaboradores ativos estão distribuídos por região?
<br>

3. Remuneração
- Qual é o salário médio dos colaboradores ativos?
- Quais áreas apresentam as maiores e menores médias salariais?
- Como o salário médio varia entre os níveis de senioridade?
<br>

4. Desligamentos
- Quais são os motivos de desligamento mais frequentes?
- Quais áreas registraram mais desligamentos?
- Quais níveis de senioridade registraram mais desligamentos?
- Como os desligamentos estão distribuídos entre os gestores?
- Em quais tempos de casa os desligamentos são mais frequentes?
<br>

5. Qualidade dos dados
- Quantos registros apresentam inconsistências ou precisam de validação?

<br>

> Essas análises permitem identificar distribuições, frequências, diferenças salariais e padrões iniciais de movimentação. Entretanto, a identificação de padrões não comprova, isoladamente, suas causas.

###CRIAÇÃO DE INDICADORES DE PEOPLE ANALYTICS

Diante dos resultados obtidos com a análise exploratória inicial, iniciaremos a construção de indicadores voltados a People Analytics, utilizando consultas SQL para transformar os dados disponíveis em informações que apoiem a tomada de decisão.

###1 - HEADCOUNT TOTAL

- Identifica a quantidade de colaboradores ativos na organização, considerando o status registrado na base de dados.

>**Pergunta de negócio:** quantos colaboradores estão ativos na base analisada?

In [61]:
# Headcount Total
query = """
SELECT
    COUNT(ID_Colaborador) AS Headcount_Atual
FROM colaboradores
WHERE Status = 'Ativo';
"""

resultado = pd.read_sql_query(query, con)
resultado

,Headcount_Atual
0,336


Como funciona:
- SELECT: define o que queremos consultar.
- COUNT(ID_Colaborador): conta os identificadores dos colaboradores.
- FROM colaboradores: informa a tabela consultada.
- WHERE Status = 'Ativo': considera somente os colaboradores ativos.
<br>

RESULTADO HEADCOUNT TOTAL

- A consulta SQL identificou **336 colaboradores ativos** na base de dados analisada. O Headcount atual considera exclusivamente os colaboradores cujo campo `Status` está preenchido como `Ativo`.

>**Interpretação para People Analytics:** o Headcount estabelece uma referência inicial para acompanhar a evolução do quadro de colaboradores e serve como informação de apoio para análises futuras, como turnover e movimentação de pessoas.

###2 - CLASSIFICAÇÃO DE DESLIGAMENTOS: VOLUNTÁRIOS E INVOLUNTÁRIOS

- Classifica os desligamentos conforme o motivo registrado na base, agrupando-os em voluntários, involuntários ou pendentes de validação.

> **Pergunta de negócio:** quantos desligamentos foram voluntários e quantos foram involuntários?

In [62]:
#Classificação de Desligamentos
query = """
SELECT
    CASE
        WHEN Motivo_Desligamento IN (
            'Pedido de demissão',
            'Mudança de carreira',
            'Salário'
        ) THEN 'Voluntário'

        WHEN Motivo_Desligamento IN (
            'Desempenho',
            'Reestruturação'
        ) THEN 'Involuntário'

        ELSE 'A validar'
    END AS Tipo_Desligamento,
    COUNT(*) AS Quantidade
FROM colaboradores
WHERE Status = 'Desligado'
GROUP BY Tipo_Desligamento
ORDER BY Quantidade DESC;
"""

resultado = pd.read_sql_query(query, con)
resultado

,Tipo_Desligamento,Quantidade
0,Voluntário,118
1,A validar,18
2,Involuntário,17


Como funciona:

- **CASE WHEN:** define as regras para classificar os motivos de desligamento.
- **WHEN ... IN (...):** verifica se o motivo está entre os valores definidos para cada categoria.
- **THEN:** atribui a classificação correspondente: `Voluntário` ou `Involuntário`.
- **ELSE 'A validar':** classifica os motivos que não se enquadram nas regras definidas.
- **COUNT(*):** conta os registros de desligamento em cada categoria.
- **FROM colaboradores:** informa a tabela consultada.
- **WHERE Status = 'Desligado':** considera somente os colaboradores registrados como desligados.
- **GROUP BY Tipo_Desligamento:** agrupa os registros conforme a classificação atribuída.
- **ORDER BY Quantidade DESC:** apresenta as categorias da maior para a menor quantidade.
<br>


RESULTADO CLASSIFICAÇÃO DOS DESLIGAMENTOS

A consulta SQL classificou os **153 registros de colaboradores desligados** da seguinte forma:

| Tipo de desligamento | Quantidade |
|---|---:|
| Voluntário | 118 |
| A validar | 18 |
| Involuntário | 17 |
| **Total** | **153** |

> **Interpretação para People Analytics:** a classificação inicial identificou 118 desligamentos voluntários, 17 involuntários e 18 casos pendentes de validação. Os resultados permitem visualizar a distribuição dos desligamentos conforme as regras definidas para o projeto.

###3 - SALDO DE MOVIMENTAÇÃO

- Identifica a diferença entre o número de admissões e desligamentos em determinado período, permitindo observar se houve crescimento ou redução líquida da movimentação de pessoas.

> **Pergunta de negócio:** entraram mais pessoas do que saíram em cada ano?

<br>

>Como interpretar o resultado
>- **Saldo positivo:** houve mais admissões do que desligamentos.
>- **Saldo negativo:** houve mais desligamentos do que admissões.
>- **Saldo zero:** o número de admissões foi igual ao de desligamentos.

In [63]:
# Saldo Movimentações
query = """
WITH admissoes AS (
    SELECT
        CAST(strftime('%Y', Data_Admissao) AS INTEGER) AS Ano,
        COUNT(*) AS Total_Admissoes
    FROM colaboradores
    WHERE Data_Admissao IS NOT NULL
    GROUP BY Ano
),
desligamentos AS (
    SELECT
        CAST(strftime('%Y', Data_Desligamento) AS INTEGER) AS Ano,
        COUNT(*) AS Total_Desligamentos
    FROM colaboradores
    WHERE Data_Desligamento IS NOT NULL
    GROUP BY Ano
),
movimentacao AS (
    SELECT Ano FROM admissoes
    UNION
    SELECT Ano FROM desligamentos
)
SELECT
    m.Ano,
    COALESCE(a.Total_Admissoes, 0) AS Total_Admissoes,
    COALESCE(d.Total_Desligamentos, 0) AS Total_Desligamentos,
    COALESCE(a.Total_Admissoes, 0)
        - COALESCE(d.Total_Desligamentos, 0) AS Saldo_Movimentacao
FROM movimentacao m
LEFT JOIN admissoes a ON m.Ano = a.Ano
LEFT JOIN desligamentos d ON m.Ano = d.Ano
ORDER BY m.Ano;
"""

resultado = pd.read_sql_query(query, con)
resultado

,Ano,Total_Admissoes,Total_Desligamentos,Saldo_Movimentacao
0,2020,0,1,-1
1,2021,116,4,112
2,2022,87,7,80
3,2023,89,19,70
4,2024,86,22,64
5,2025,122,32,90
6,2026,0,22,-22
7,2027,0,31,-31
8,2028,0,10,-10
9,2029,0,5,-5


Como funciona:

- **WITH admissoes AS:** cria uma consulta temporária com o total de admissões por ano.
- **WITH desligamentos AS:** organiza o total de desligamentos por ano.
- **strftime('%Y', Data_Admissao / Data_Desligamento):** extrai o ano de cada data.
- **COUNT(*):** conta os registros de cada grupo.
- **GROUP BY Ano:** agrupa as admissões e os desligamentos por ano.
- **UNION:** reúne os anos encontrados nas duas consultas, eliminando duplicidades.
- **LEFT JOIN:** combina as admissões e os desligamentos correspondentes a cada ano.
- **COALESCE(..., 0):** substitui valores nulos por zero.
- **Saldo_Movimentacao:** calcula a diferença entre admissões e desligamentos.
- **ORDER BY m.Ano:** organiza os resultados por ano.

<br>

RESULTADO SALDO DE MOVIMENTAÇÃO

A consulta SQL identificou o saldo anual entre admissões e desligamentos, permitindo observar períodos com maior entrada ou saída de colaboradores.

> **Interpretação para People Analytics:** entre 2021 e 2025, o saldo foi positivo, indicando mais admissões do que desligamentos. Em 2026, 2027, 2028 e 2029, o saldo foi negativo, pois foram registrados desligamentos sem admissões na base para esses anos.

- O resultado de 2020 também apresentou saldo negativo de 1, devido ao registro de um desligamento e nenhuma admissão.

### 4 - TURNOVER GERAL

- Calcula a taxa de rotatividade dos colaboradores por ano, relacionando os desligamentos ocorridos no período ao headcount médio estimado.

> **Pergunta de negócio:** qual foi a taxa de turnover da organização em cada ano analisado?

<br>


**COMO INTERPRETAR O RESULTADO**

- **Taxa elevada:** indica maior rotatividade e pode justificar uma investigação das causas dos desligamentos.
- **Taxa baixa:** indica menor rotatividade proporcional ao quadro médio de colaboradores.
- **Taxa zero:** indica que não houve desligamentos no período analisado.

>A análise considera os anos completos de 2021 a 2025, evitando incluir os registros de anos futuros da base simulada.

In [65]:
# Turnover Geral por Ano
query = """
WITH anos AS (
    SELECT 2021 AS Ano
    UNION ALL SELECT 2022
    UNION ALL SELECT 2023
    UNION ALL SELECT 2024
    UNION ALL SELECT 2025
),
headcount AS (
    SELECT
        a.Ano,

        SUM(
            CASE
                WHEN c.Data_Admissao < printf('%04d-01-01', a.Ano)
                 AND (
                     c.Data_Desligamento IS NULL
                     OR c.Data_Desligamento >= printf('%04d-01-01', a.Ano)
                 )
                THEN 1 ELSE 0
            END
        ) AS Headcount_Inicial,

        SUM(
            CASE
                WHEN c.Data_Admissao <= printf('%04d-12-31', a.Ano)
                 AND (
                     c.Data_Desligamento IS NULL
                     OR c.Data_Desligamento > printf('%04d-12-31', a.Ano)
                 )
                THEN 1 ELSE 0
            END
        ) AS Headcount_Final

    FROM anos a
    CROSS JOIN colaboradores c
    GROUP BY a.Ano
),
desligamentos AS (
    SELECT
        CAST(strftime('%Y', Data_Desligamento) AS INTEGER) AS Ano,
        COUNT(*) AS Total_Desligamentos
    FROM colaboradores
    WHERE Data_Desligamento IS NOT NULL
      AND CAST(strftime('%Y', Data_Desligamento) AS INTEGER)
          BETWEEN 2021 AND 2025
    GROUP BY Ano
)
SELECT
    h.Ano,
    h.Headcount_Inicial,
    h.Headcount_Final,
    ROUND(
        (h.Headcount_Inicial + h.Headcount_Final) / 2.0,
        2
    ) AS Headcount_Medio,
    COALESCE(d.Total_Desligamentos, 0) AS Total_Desligamentos,
    ROUND(
        COALESCE(d.Total_Desligamentos, 0) * 100.0
        / NULLIF(
            (h.Headcount_Inicial + h.Headcount_Final) / 2.0,
            0
        ),
        2
    ) AS Turnover_Percentual
FROM headcount h
LEFT JOIN desligamentos d ON h.Ano = d.Ano
ORDER BY h.Ano;
"""

resultado = pd.read_sql_query(query, con)
resultado

,Ano,Headcount_Inicial,Headcount_Final,Headcount_Medio,Total_Desligamentos,Turnover_Percentual
0,2021,0,112,56.0,4,7.14
1,2022,113,191,152.0,7,4.61
2,2023,191,262,226.5,19,8.39
3,2024,262,325,293.5,22,7.50
4,2025,326,415,370.5,32,8.64


**Como funciona:**
- `anos`: define o período analisado, de 2021 a 2025.
- `headcount`: calcula a quantidade estimada de colaboradores no início e no final de cada ano.
- `desligamentos`: conta os desligamentos registrados em cada ano do período.
- `COALESCE`: considera zero quando não há desligamentos registrados naquele ano.
- `Headcount_Medio`: calcula a média entre o headcount inicial e final.
- `NULLIF`: evita divisão por zero.
- `Turnover_Percentual`: calcula a taxa de desligamento em relação ao headcount médio.
- `LEFT JOIN`: reúne as informações de headcount e desligamentos por ano.
- `ORDER BY`: organiza os resultados cronologicamente.

<br>

RESULTADO TURNOVER GERAL

A consulta SQL calculou a taxa de turnover anual entre 2021 e 2025, relacionando o total de desligamentos ao headcount médio estimado de cada período.

<br>

**Interpretação para People Analytics:**
- **2021:** apresentou turnover de 7,14%, considerando a premissa de início da operação sem colaboradores e o crescimento do quadro ao longo do ano.
- **2022:** registrou a menor taxa do período, com 4,61%.
- **2023:** a taxa aumentou para 8,39%, indicando maior rotatividade proporcional.
- **2024:** apresentou redução para 7,50%.
- **2025:** registrou a maior taxa do período, com 8,64%, sendo um ano que merece atenção em uma investigação das causas dos desligamentos.

>**Conclusão:** entre 2021 e 2025, a maior taxa de turnover ocorreu em 2025, enquanto a menor foi registrada em 2022. A análise permite acompanhar a evolução da rotatividade e identificar períodos que merecem investigação.

### 5 - TURNOVER VOLUNTÁRIO

Calcula a taxa anual de desligamentos voluntários, considerando os motivos classificados como iniciativa do colaborador.

> **Pergunta de negócio:** qual foi a taxa de turnover voluntário da organização em cada ano analisado?

<br>

**COMO INTERPRETAR O RESULTADO**

- **Taxa elevada:** pode indicar dificuldades de retenção e exige investigação dos motivos das saídas.
- **Taxa baixa:** indica menor rotatividade voluntária proporcional ao quadro médio.
- **Taxa zero:** indica ausência de desligamentos voluntários no período analisado.

>A análise considera os anos completos de 2021 a 2025. Os casos classificados como “A validar” não serão contabilizados como desligamentos voluntários ou involuntários.

In [66]:
# Turnover Voluntário por Ano
query = """
WITH headcount AS (
    SELECT
        Ano,
        (Headcount_Inicial + Headcount_Final) / 2.0
            AS Headcount_Medio
    FROM (
        SELECT
            CAST(strftime('%Y', c.Data_Admissao) AS INTEGER) AS Ano,
            SUM(
                CASE
                    WHEN c.Data_Admissao < printf(
                        '%04d-01-01',
                        CAST(strftime('%Y', c.Data_Admissao) AS INTEGER)
                    )
                    THEN 1 ELSE 0
                END
            ) AS Headcount_Inicial,
            COUNT(*) AS Headcount_Final
        FROM colaboradores c
        WHERE 1 = 0
        GROUP BY Ano
    )
),
anos AS (
    SELECT 2021 AS Ano
    UNION ALL SELECT 2022
    UNION ALL SELECT 2023
    UNION ALL SELECT 2024
    UNION ALL SELECT 2025
),
headcount_anual AS (
    SELECT
        a.Ano,
        SUM(
            CASE
                WHEN c.Data_Admissao < printf('%04d-01-01', a.Ano)
                 AND (
                     c.Data_Desligamento IS NULL
                     OR c.Data_Desligamento >= printf('%04d-01-01', a.Ano)
                 )
                THEN 1 ELSE 0
            END
        ) AS Headcount_Inicial,
        SUM(
            CASE
                WHEN c.Data_Admissao <= printf('%04d-12-31', a.Ano)
                 AND (
                     c.Data_Desligamento IS NULL
                     OR c.Data_Desligamento > printf('%04d-12-31', a.Ano)
                 )
                THEN 1 ELSE 0
            END
        ) AS Headcount_Final
    FROM anos a
    CROSS JOIN colaboradores c
    GROUP BY a.Ano
),
voluntarios AS (
    SELECT
        CAST(strftime('%Y', Data_Desligamento) AS INTEGER) AS Ano,
        COUNT(*) AS Total_Voluntarios
    FROM colaboradores
    WHERE Status = 'Desligado'
      AND Motivo_Desligamento IN (
          'Pedido de demissão',
          'Mudança de carreira',
          'Salário'
      )
      AND CAST(strftime('%Y', Data_Desligamento) AS INTEGER)
          BETWEEN 2021 AND 2025
    GROUP BY Ano
)
SELECT
    h.Ano,
    COALESCE(v.Total_Voluntarios, 0) AS Total_Voluntarios,
    ROUND(
        (h.Headcount_Inicial + h.Headcount_Final) / 2.0,
        2
    ) AS Headcount_Medio,
    ROUND(
        COALESCE(v.Total_Voluntarios, 0) * 100.0
        / NULLIF(
            (h.Headcount_Inicial + h.Headcount_Final) / 2.0,
            0
        ),
        2
    ) AS Turnover_Voluntario_Percentual
FROM headcount_anual h
LEFT JOIN voluntarios v ON h.Ano = v.Ano
ORDER BY h.Ano;
"""

resultado = pd.read_sql_query(query, con)
resultado

,Ano,Total_Voluntarios,Headcount_Medio,Turnover_Voluntario_Percentual
0,2021,3,56.0,5.36
1,2022,6,152.0,3.95
2,2023,14,226.5,6.18
3,2024,16,293.5,5.45
4,2025,24,370.5,6.48


**Como funciona:**
- anos: define o período de 2021 a 2025.
- headcount_anual: calcula o headcount inicial e final de cada ano.
- voluntarios: conta os desligamentos classificados como voluntários em cada ano.
- LEFT JOIN: reúne os dados de headcount e desligamentos.
- NULLIF: evita divisão por zero.
- Turnover_Voluntario_Percentual: apresenta a taxa anual de turnover voluntário.
<br>

RESULTADO TURNOVER VOLUNTÁRIO

A consulta SQL calculou a taxa anual de turnover voluntário entre 2021 e 2025, considerando os desligamentos classificados como voluntários e o headcount médio estimado de cada ano.

<br>

**Interpretação para People Analytics:**

- **2021:** apresentou turnover voluntário de 5,36%.
- **2022:** registrou a menor taxa do período, com 3,95%.
- **2023:** a taxa aumentou para 6,18%.
- **2024:** houve redução para 5,45%.
- **2025:** registrou a maior taxa do período, com 6,48%.

>**Conclusão:** a maior taxa de turnover voluntário ocorreu em 2025, enquanto a menor foi registrada em 2022. Essa variação pode orientar uma investigação sobre os motivos das saídas voluntárias ao longo do período.

### 6 - TURNOVER INVOLUNTÁRIO

Calcula a taxa anual de desligamentos involuntários, considerando os motivos classificados como iniciativa da organização.

> **Pergunta de negócio:** qual foi a taxa de turnover involuntário da organização em cada ano analisado?

<br>

**COMO INTERPRETAR O RESULTADO**

- **Taxa elevada:** pode indicar aumento de desligamentos por iniciativa da organização e justificar uma investigação das causas.
- **Taxa baixa:** indica menor rotatividade involuntária proporcional ao quadro médio.
- **Taxa zero:** indica ausência de desligamentos involuntários no período analisado.

>A análise considera os anos completos de 2021 a 2025. Os casos classificados como “A validar” não serão contabilizados como desligamentos involuntários.

In [67]:
# Turnover Involuntário por Ano
query = """
WITH anos AS (
    SELECT 2021 AS Ano
    UNION ALL SELECT 2022
    UNION ALL SELECT 2023
    UNION ALL SELECT 2024
    UNION ALL SELECT 2025
),
headcount_anual AS (
    SELECT
        a.Ano,
        SUM(
            CASE
                WHEN c.Data_Admissao < printf('%04d-01-01', a.Ano)
                 AND (
                     c.Data_Desligamento IS NULL
                     OR c.Data_Desligamento >= printf('%04d-01-01', a.Ano)
                 )
                THEN 1 ELSE 0
            END
        ) AS Headcount_Inicial,
        SUM(
            CASE
                WHEN c.Data_Admissao <= printf('%04d-12-31', a.Ano)
                 AND (
                     c.Data_Desligamento IS NULL
                     OR c.Data_Desligamento > printf('%04d-12-31', a.Ano)
                 )
                THEN 1 ELSE 0
            END
        ) AS Headcount_Final
    FROM anos a
    CROSS JOIN colaboradores c
    GROUP BY a.Ano
),
involuntarios AS (
    SELECT
        CAST(strftime('%Y', Data_Desligamento) AS INTEGER) AS Ano,
        COUNT(*) AS Total_Involuntarios
    FROM colaboradores
    WHERE Status = 'Desligado'
      AND Motivo_Desligamento IN (
          'Desempenho',
          'Reestruturação'
      )
      AND CAST(strftime('%Y', Data_Desligamento) AS INTEGER)
          BETWEEN 2021 AND 2025
    GROUP BY Ano
)
SELECT
    h.Ano,
    COALESCE(i.Total_Involuntarios, 0) AS Total_Involuntarios,
    ROUND(
        (h.Headcount_Inicial + h.Headcount_Final) / 2.0,
        2
    ) AS Headcount_Medio,
    ROUND(
        COALESCE(i.Total_Involuntarios, 0) * 100.0
        / NULLIF(
            (h.Headcount_Inicial + h.Headcount_Final) / 2.0,
            0
        ),
        2
    ) AS Turnover_Involuntario_Percentual
FROM headcount_anual h
LEFT JOIN involuntarios i ON h.Ano = i.Ano
ORDER BY h.Ano;
"""

resultado = pd.read_sql_query(query, con)
resultado

,Ano,Total_Involuntarios,Headcount_Medio,Turnover_Involuntario_Percentual
0,2021,1,56.0,1.79
1,2022,1,152.0,0.66
2,2023,3,226.5,1.32
3,2024,3,293.5,1.02
4,2025,4,370.5,1.08


**Como funciona:**
- `anos`: define o período de análise de 2021 a 2025.
- `headcount_anual`: calcula o headcount inicial e final de cada ano.
- `involuntarios`: conta os desligamentos classificados como involuntários por ano.
- `COALESCE`: retorna zero quando não há desligamentos involuntários no ano.
- `LEFT JOIN`: combina os desligamentos com o headcount anual.
- `NULLIF`: evita divisão por zero.
- `Turnover_Involuntario_Percentual`: apresenta a taxa anual de turnover involuntário.

### RESULTADO TURNOVER INVOLUNTÁRIO

A consulta SQL calculou a taxa anual de turnover involuntário entre 2021 e 2025, considerando os desligamentos classificados como involuntários e o headcount médio estimado de cada ano.

<br>

**Interpretação para People Analytics:**

- **2021:** apresentou turnover involuntário de 1,79%.
- **2022:** registrou a menor taxa do período, com 0,66%.
- **2023:** a taxa aumentou para 1,32%.
- **2024:** apresentou redução para 1,02%.
- **2025:** registrou uma taxa de 1,08%, apesar de apresentar o maior número de desligamentos involuntários, com quatro casos.

>**Conclusão:** a maior taxa de turnover involuntário ocorreu em 2021, enquanto a menor foi registrada em 2022. Em 2025, houve o maior número absoluto de desligamentos involuntários do período, mas a taxa permaneceu próxima à de 2024.

### 7 - RETENÇÃO DE PESSOAS

Calcula o percentual de colaboradores que faziam parte do quadro inicial e permaneceram na organização até o final do período analisado.

> **Pergunta de negócio:** qual percentual dos colaboradores presentes no início do ano permaneceu na organização até o final desse mesmo ano?

<br>

**COMO INTERPRETAR O RESULTADO**

- **Retenção elevada:** indica que uma proporção maior dos colaboradores iniciais permaneceu na organização.
- **Retenção baixa:** indica que uma proporção menor dos colaboradores iniciais permaneceu até o final do período.
- **Retenção de 100%:** indica que todos os colaboradores do quadro inicial permaneceram até o final do período.
- **Retenção de 0%:** indica que nenhum colaborador do quadro inicial permaneceu até o final do período.

>A análise considera os anos de 2021 a 2025. O cálculo utiliza as datas de admissão e desligamento registradas na base simulada.

In [68]:
# Retenção de Pessoas por Ano
query = """
WITH anos AS (
    SELECT 2021 AS Ano
    UNION ALL SELECT 2022
    UNION ALL SELECT 2023
    UNION ALL SELECT 2024
    UNION ALL SELECT 2025
),
headcount AS (
    SELECT
        a.Ano,
        SUM(
            CASE
                WHEN c.Data_Admissao < printf('%04d-01-01', a.Ano)
                 AND (
                     c.Data_Desligamento IS NULL
                     OR c.Data_Desligamento >= printf('%04d-01-01', a.Ano)
                 )
                THEN 1 ELSE 0
            END
        ) AS Headcount_Inicial,
        SUM(
            CASE
                WHEN c.Data_Admissao < printf('%04d-01-01', a.Ano)
                 AND (
                     c.Data_Desligamento IS NULL
                     OR c.Data_Desligamento > printf('%04d-12-31', a.Ano)
                 )
                THEN 1 ELSE 0
            END
        ) AS Colaboradores_Retidos
    FROM anos a
    CROSS JOIN colaboradores c
    GROUP BY a.Ano
)
SELECT
    Ano,
    Headcount_Inicial,
    Colaboradores_Retidos,
    ROUND(
        Colaboradores_Retidos * 100.0
        / NULLIF(Headcount_Inicial, 0),
        2
    ) AS Retencao_Percentual
FROM headcount
ORDER BY Ano;
"""

resultado = pd.read_sql_query(query, con)
resultado

,Ano,Headcount_Inicial,Colaboradores_Retidos,Retencao_Percentual
0,2021,0,0,NaN
1,2022,113,108,95.58
2,2023,191,175,91.62
3,2024,262,245,93.51
4,2025,326,298,91.41


**Como funciona:**
- `anos`: define o período de análise entre 2021 e 2025.
- `headcount`: identifica os colaboradores que já estavam na organização no início de cada ano e quantos deles permaneceram até o final.
- `Headcount_Inicial`: conta os colaboradores admitidos antes do início do ano e ainda vinculados nessa data.
- `Colaboradores_Retidos`: conta os colaboradores do quadro inicial que não tinham desligamento até o final do ano.
- `NULLIF`: evita divisão por zero.
- `Retencao_Percentual`: apresenta o percentual de colaboradores iniciais que permaneceram na organização.
<br>

### RESULTADO RETENÇÃO DE PESSOAS

A consulta SQL calculou o percentual de colaboradores que faziam parte do quadro inicial e permaneceram na organização até o final de cada ano, entre 2021 e 2025.

<br>


**Interpretação para People Analytics:**

- **2021:** não foi possível calcular a taxa, pois o headcount inicial era zero, considerando a premissa de início da operação nesse ano.
- **2022:** apresentou a maior retenção do período, com 95,58%.
- **2023:** a retenção caiu para 91,62%.
- **2024:** houve recuperação para 93,51%.
- **2025:** a retenção foi de 91,41%, a segunda menor taxa entre os anos calculáveis.

>**Conclusão:** entre 2022 e 2025, a retenção permaneceu acima de 90%, mas apresentou oscilações. A menor taxa ocorreu em 2025, indicando que uma proporção menor dos colaboradores presentes no início do ano permaneceu até o final.

### 8 - TAXA DE DESLIGAMENTO POR ÁREA

Calcula a proporção de desligamentos em relação ao headcount médio de cada área, permitindo comparar a rotatividade entre diferentes setores da organização.

> **Pergunta de negócio:** quais áreas apresentam as maiores taxas de desligamento no período analisado?

<br>

**COMO INTERPRETAR O RESULTADO**

- **Taxa elevada:** indica maior proporção de desligamentos em relação ao quadro médio da área.
- **Taxa baixa:** indica menor proporção de desligamentos em relação ao quadro médio da área.
- **Taxa zero:** indica que não foram registrados desligamentos na área durante o período.

>A análise considera os anos completos de 2021 a 2025 e utiliza as datas de admissão e desligamento registradas na base simulada.

In [69]:
# Taxa de Desligamento por Área
query = """
WITH anos AS (
    SELECT 2021 AS Ano
    UNION ALL SELECT 2022
    UNION ALL SELECT 2023
    UNION ALL SELECT 2024
    UNION ALL SELECT 2025
),
headcount_anual AS (
    SELECT
        a.Ano,
        c.Area,
        SUM(
            CASE
                WHEN c.Data_Admissao < printf('%04d-01-01', a.Ano)
                 AND (
                     c.Data_Desligamento IS NULL
                     OR c.Data_Desligamento >= printf('%04d-01-01', a.Ano)
                 )
                THEN 1 ELSE 0
            END
        ) AS Headcount_Inicial,
        SUM(
            CASE
                WHEN c.Data_Admissao <= printf('%04d-12-31', a.Ano)
                 AND (
                     c.Data_Desligamento IS NULL
                     OR c.Data_Desligamento > printf('%04d-12-31', a.Ano)
                 )
                THEN 1 ELSE 0
            END
        ) AS Headcount_Final
    FROM anos a
    CROSS JOIN colaboradores c
    GROUP BY a.Ano, c.Area
),
desligamentos AS (
    SELECT
        CAST(strftime('%Y', Data_Desligamento) AS INTEGER) AS Ano,
        Area,
        COUNT(*) AS Total_Desligamentos
    FROM colaboradores
    WHERE Data_Desligamento IS NOT NULL
      AND CAST(strftime('%Y', Data_Desligamento) AS INTEGER)
          BETWEEN 2021 AND 2025
    GROUP BY Ano, Area
)
SELECT
    h.Ano,
    h.Area,
    COALESCE(d.Total_Desligamentos, 0) AS Total_Desligamentos,
    ROUND(
        (h.Headcount_Inicial + h.Headcount_Final) / 2.0,
        2
    ) AS Headcount_Medio,
    ROUND(
        COALESCE(d.Total_Desligamentos, 0) * 100.0
        / NULLIF(
            (h.Headcount_Inicial + h.Headcount_Final) / 2.0,
            0
        ),
        2
    ) AS Taxa_Desligamento_Percentual
FROM headcount_anual h
LEFT JOIN desligamentos d
    ON h.Ano = d.Ano
    AND h.Area = d.Area
ORDER BY h.Ano, Taxa_Desligamento_Percentual DESC;
"""

resultado = pd.read_sql_query(query, con)
resultado

,Ano,Area,Total_Desligamentos,Headcount_Medio,Taxa_Desligamento_Percentual
0,2021,Produto,1,5.5,18.18
1,2021,Comercial,1,7.0,14.29
2,2021,Dados,1,7.0,14.29
3,2021,Operações,1,7.0,14.29
4,2021,Financeiro,0,5.0,0.00
5,2021,Jurídico,0,5.0,0.00
6,2021,Marketing,0,8.0,0.00
7,2021,Recursos Humanos,0,8.0,0.00
8,2021,Tecnologia,0,3.5,0.00
9,2022,Recursos Humanos,4,18.0,22.22


**Como funciona:**
- `anos`: define o período de análise entre 2021 e 2025.
- `headcount_anual`: estima o headcount inicial e final de cada área por ano.
- `desligamentos`: conta os desligamentos registrados por área e ano.
- `LEFT JOIN`: combina os dados de headcount e desligamentos.
- `COALESCE`: considera zero quando não há desligamentos registrados.
- `NULLIF`: evita divisão por zero.
- `Taxa_Desligamento_Percentual`: calcula a proporção de desligamentos em relação ao headcount médio estimado de cada área.
- `ORDER BY`: organiza os resultados por ano e pelas maiores taxas.

### RESULTADO TAXA DE DESLIGAMENTO POR ÁREA

A consulta SQL calculou a taxa anual de desligamentos por área entre 2021 e 2025, relacionando os desligamentos registrados ao headcount médio estimado de cada setor.

<br>

**Interpretação para People Analytics:**

- **2021:** Produto apresentou a maior taxa, com 18,18%. O resultado deve ser interpretado considerando o início da operação e o baixo headcount estimado das áreas.
- **2022:** Recursos Humanos apresentou a maior taxa, com 22,22%.
- **2023:** Financeiro registrou a maior taxa, com 14,63%, seguido de Marketing, com 13,79%.
- **2024:** Financeiro apresentou a maior taxa, com 15,38%, seguido de Tecnologia, com 11,76%.
- **2025:** Tecnologia registrou a maior taxa, com 14,93%, seguida de Recursos Humanos, com 12,35%, e Dados, com 11,76%.

>**Conclusão:** as áreas com maiores taxas de desligamento variaram ao longo do período. Em 2025, Tecnologia apresentou a maior taxa, enquanto Financeiro se destacou em 2023 e 2024. Esses resultados ajudam a direcionar investigações sobre possíveis causas de rotatividade em cada setor.

### CONSIDERAÇÕES FINAIS

- A exploração da Base 3 por meio de SQL permitiu responder perguntas de negócio relacionadas à estrutura organizacional, à remuneração e à movimentação de pessoas.

>As consultas foram executadas com a base carregada em SQLite e referenciada como `colaboradores`, sem alteração do arquivo original.

<br>

Entre as perguntas respondidas, destacam-se:

- **Headcount:** quantos colaboradores estão ativos?
- **Movimentação:** quantas admissões e desligamentos ocorreram por ano? O saldo de movimentação foi positivo ou negativo?
- **Composição organizacional:** como os colaboradores estão distribuídos por área, cargo, senioridade e região?
- **Remuneração:** qual é o salário médio geral e como ele varia por área e senioridade?
- **Desligamentos:** quais são os motivos mais frequentes e como as saídas se distribuem por área, senioridade e gestor?
- **Tempo de casa:** quanto tempo os colaboradores permaneceram na organização antes do desligamento?
- **Turnover e retenção:** quais foram as taxas de turnover geral, voluntário e involuntário, a retenção anual e as taxas de desligamento por área?
- **Qualidade dos dados:** quantos registros apresentam possíveis inconsistências e quais problemas foram identificados?

As análises permitiram identificar padrões e levantar hipóteses para investigações futuras. Entretanto, os resultados de turnover, retenção e desligamentos por área ainda exigem validação dos critérios temporais e das regras de contagem utilizadas.

>**Evolução Futura:** recomenda-se: calcular a dispersão salarial e o tempo médio de casa dos colaboradores ativos, além de aprofundar análises de equidade salarial. Indicadores de absenteísmo, tempo de preenchimento de vagas, custo de contratação, engajamento, desempenho e treinamento dependerão da disponibilidade de novas informações.